In [3]:
from transformers import AutoTokenizer

# 1. Define your Jinja2 chat template string
# This template will format user messages with "Question:"
# and then, if add_generation_prompt=True, it will append the "Answer: Let's think step by step." part.
jinja_chat_template = (
    "{% for message in messages %}"
        "{% if message.role == 'user' %}"
            "Question: {{ message.content }}\n\n"
        "{% elif message.role == 'assistant' %}"
            "Answer: {{ message.content }}\n\n"  # How to format previous assistant messages, if any
        "{% elif message.role == 'system' %}" # Optional: How to handle system messages
            "{{ message.content }}\n\n"
        "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}" # The key part for your request
        "{% if messages[-1].role != 'assistant' %}" # Only add if last message isn't already assistant
            "Answer: Let's think step by step.\n"
        "{% endif %}"
    "{% endif %}"
)

# 2. Load a pretrained tokenizer
# For demonstration, let's use a common one. Replace with your model's tokenizer.
# If your tokenizer already has a chat_template, this will override it.
tokenizer_name = "Qwen/Qwen2.5-0.5B" # Example
# tokenizer_name = "gpt2" # Another example, gpt2 doesn't have a default chat template
try:
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
except Exception as e:
    print(f"Could not load {tokenizer_name}, trying gpt2 as a fallback: {e}")
    tokenizer = AutoTokenizer.from_pretrained("gpt2")


# 3. Set the chat_template for the tokenizer
tokenizer.chat_template = jinja_chat_template

# Ensure a pad token is set if it's not (good practice, though not strictly for this string formatting)
if tokenizer.pad_token is None:
    if tokenizer.eos_token is not None:
        print(f"Tokenizer {tokenizer_name} has no pad_token, setting it to eos_token: {tokenizer.eos_token}")
        tokenizer.pad_token = tokenizer.eos_token
    else:
        # Add a generic pad token if eos is also missing, e.g. for gpt2
        print(f"Tokenizer {tokenizer_name} has no pad_token and no eos_token. Adding [PAD] as pad_token.")
        tokenizer.add_special_tokens({'pad_token': '[PAD]'})


# 4. Prepare your chat messages
# For your specific prompt, it seems you'll usually have a single user message.
chat_messages = [
    {"role": "user", "content": "What are the main challenges in developing large language models?"}
]

# 5. Apply the chat template
# - tokenize=False: Returns the formatted string instead of token IDs.
# - add_generation_prompt=True: Crucial for appending the "Answer: Let's think step by step." part.
formatted_prompt_string = tokenizer.apply_chat_template(
    chat_messages,
    tokenize=False,
    add_generation_prompt=True
)

# 6. Print the result
print("--- Formatted Prompt String ---")
print(formatted_prompt_string)

# --- Verification ---
expected_output = """Question: What are the main challenges in developing large language models?

Answer: Let's think step by step.
"""
assert formatted_prompt_string == expected_output
print("\n--- Verification Successful ---")
print("The output matches the desired format.")


# Example with a system message and a previous assistant turn (to show template flexibility)
# Note: Your specific "Answer: Let's think step by step." is a generation prompt,
# so it's typically added when you want the model to start generating from that point.
chat_with_history = [
    {"role": "system", "content": "You are a helpful AI assistant focused on deep learning."},
    {"role": "user", "content": "What is a Transformer?"},
    {"role": "assistant", "content": "A Transformer is a deep learning model architecture."},
    {"role": "user", "content": "And what are its main challenges for LLMs?"}
]

formatted_with_history = tokenizer.apply_chat_template(
    chat_with_history,
    tokenize=False,
    add_generation_prompt=True
)
print("\n--- Formatted Prompt String with History ---")
print(formatted_with_history)

expected_output_history = """You are a helpful AI assistant focused on deep learning.

Question: What is a Transformer?

Answer: A Transformer is a deep learning model architecture.

Question: And what are its main challenges for LLMs?

Answer: Let's think step by step.
"""
assert formatted_with_history == expected_output_history
print("\n--- Verification with History Successful ---")

--- Formatted Prompt String ---
Question: What are the main challenges in developing large language models?

Answer: Let's think step by step.


--- Verification Successful ---
The output matches the desired format.

--- Formatted Prompt String with History ---
You are a helpful AI assistant focused on deep learning.

Question: What is a Transformer?

Answer: A Transformer is a deep learning model architecture.

Question: And what are its main challenges for LLMs?

Answer: Let's think step by step.


--- Verification with History Successful ---


In [5]:
from transformers import AutoTokenizer

# Define the Jinja2 chat template string as above
jinja_chat_template = (
    "{% for message in messages %}"
        "{% if message.role == 'user' %}"
            "Question: {{ message.content }}\n\n"
            "Answer: Let's think step by step.\n"
        "{% elif message.role == 'assistant' %}"
            "{{ message.content }}\n"
        "{% elif message.role == 'system' %}"
            "{{ message.content }}\n\n"
        "{% endif %}"
    "{% endfor %}"
)

# Load a pretrained tokenizer.
# Using gpt2 as it has no default chat_template, making it a clean slate.
# If you use a tokenizer like mistral-instruct, this will override its native template.
tokenizer_name = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

# Set the custom chat_template
tokenizer.chat_template = jinja_chat_template

# Ensure a pad token is set (good practice, especially if tokenizing)
if tokenizer.pad_token is None:
    if tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token
        print(f"Set pad_token to eos_token: {tokenizer.pad_token}")
    else:
        # Add a generic pad token if eos is also missing
        new_pad_token = '[PAD]'
        tokenizer.add_special_tokens({'pad_token': new_pad_token})
        # After adding, explicitly set it if the prior line doesn't automatically
        tokenizer.pad_token = new_pad_token
        print(f"Added and set new pad_token: {tokenizer.pad_token}")


# --- Test Case 1: Single user message (for generation) ---
# This should match your PROMPT_TEMPLATE string literal.
print("\n--- Test Case 1: Single User Message (add_generation_prompt=True) ---")
# Your PROMPT_TEMPLATE, interpreted as a Python string literal:
# PROMPT_TEMPLATE = """
# Question: {question}
#
# Answer: Let's think step by step.
# """
# This string is: "\nQuestion: {question}\n\nAnswer: Let's think step by step.\n"

user_question_content = "What are the main challenges in developing large language models?"
chat_for_generation = [
    {"role": "user", "content": user_question_content}
]

# `add_generation_prompt=True` is typically used to signal readiness for the assistant's turn.
# Our template now builds the "Answer: Let's think step by step." part directly.
formatted_prompt_gen = tokenizer.apply_chat_template(
    chat_for_generation,
    tokenize=False,
    add_generation_prompt=True
)
print("Formatted string (for generation):")
print(f"'{formatted_prompt_gen}'")

expected_output_gen = (
    f"Question: {user_question_content}\n\n"
    "Answer: Let's think step by step.\n"
)
assert formatted_prompt_gen == expected_output_gen
print("Test Case 1: Correct! Matches desired PROMPT_TEMPLATE.")


# --- Test Case 2: Full conversation (user and assistant) ---
print("\n--- Test Case 2: Full Conversation (add_generation_prompt=False) ---")
assistant_answer_content = "One key challenge is the computational cost and data requirements."
full_conversation = [
    {"role": "user", "content": user_question_content},
    {"role": "assistant", "content": assistant_answer_content}
]

# `add_generation_prompt=False` is used to format an existing, complete conversation.
formatted_full_convo = tokenizer.apply_chat_template(
    full_conversation,
    tokenize=False,
    add_generation_prompt=False
)
print("Formatted string (full conversation):")
print(f"'{formatted_full_convo}'")

expected_output_full_convo = (
    f"Question: {user_question_content}\n\n"
    "Answer: Let's think step by step.\n"
    f"{assistant_answer_content}\n"
)
assert formatted_full_convo == expected_output_full_convo
print("Test Case 2: Correct! Consistent prefix, assistant content appended.")

# --- Test Case 3: Conversation with a system message ---
print("\n--- Test Case 3: System, User, Assistant (add_generation_prompt=False) ---")
system_message_content = "You are a helpful AI focusing on ML."
conversation_with_system = [
    {"role": "system", "content": system_message_content},
    {"role": "user", "content": user_question_content},
    {"role": "assistant", "content": assistant_answer_content}
]
formatted_with_system = tokenizer.apply_chat_template(
    conversation_with_system,
    tokenize=False,
    add_generation_prompt=False
)
print("Formatted string (with system message):")
print(f"'{formatted_with_system}'")
expected_output_with_system = (
    f"{system_message_content}\n\n"
    f"Question: {user_question_content}\n\n" # No extra \n here because user is not loop.first
    "Answer: Let's think step by step.\n"
    f"{assistant_answer_content}\n"
)
assert formatted_with_system == expected_output_with_system
print("Test Case 3: Correct! System message handled, no extra newlines.")

# --- Test Case 4: System and User (for generation) ---
print("\n--- Test Case 4: System, User (add_generation_prompt=True) ---")
conversation_sys_user_for_gen = [
    {"role": "system", "content": system_message_content},
    {"role": "user", "content": user_question_content}
]
formatted_sys_user_gen = tokenizer.apply_chat_template(
    conversation_sys_user_for_gen,
    tokenize=False,
    add_generation_prompt=True
)
print("Formatted string (system, user, for generation):")
print(f"'{formatted_sys_user_gen}'")
expected_output_sys_user_gen = (
    f"{system_message_content}\n\n"
    f"Question: {user_question_content}\n\n"
    "Answer: Let's think step by step.\n"
)
assert formatted_sys_user_gen == expected_output_sys_user_gen
print("Test Case 4: Correct!")


--- Test Case 1: Single User Message (add_generation_prompt=True) ---
Formatted string (for generation):
'Question: What are the main challenges in developing large language models?

Answer: Let's think step by step.
'
Test Case 1: Correct! Matches desired PROMPT_TEMPLATE.

--- Test Case 2: Full Conversation (add_generation_prompt=False) ---
Formatted string (full conversation):
'Question: What are the main challenges in developing large language models?

Answer: Let's think step by step.
One key challenge is the computational cost and data requirements.
'
Test Case 2: Correct! Consistent prefix, assistant content appended.

--- Test Case 3: System, User, Assistant (add_generation_prompt=False) ---
Formatted string (with system message):
'You are a helpful AI focusing on ML.

Question: What are the main challenges in developing large language models?

Answer: Let's think step by step.
One key challenge is the computational cost and data requirements.
'
Test Case 3: Correct! System mes